# 成分股分钟覆盖：先验证新信息是否可观察
只取Q3已披露篮子并集30只，不按收益挑股票或日期；NONE/RTH五分钟，保留缺口和异常，不训练、不生成交易。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
import exchange_calendars as xc
from src.data.minute_intake import collect_futu_bundle,atomic_json
from src.data.minute_price_basis import normalize_none_history
from src.data.semis_context_refresh import audit_context
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/constituent_intraday_q3_20261014';plan=json.loads((card/'plan.json').read_text());output=root/plan['raw_directory'];output.mkdir(parents=True,exist_ok=True)
def progress(value):atomic_json(value,Path(plan['progress_path']))
source=collect_futu_bundle(requests={s:'K_5M' for s in plan['symbols']},start=plan['start'],end=plan['as_of'],output_dir=output,checkpoint_dir=Path(plan['checkpoint_dir']),workers=1,request_pause_seconds=1.,progress=progress)
parts=[];raw_sources={}
for entry in source['entries']:
 path=output/entry['path'];assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];data=pd.read_parquet(path);raw_sources[entry['symbol']]=data;parts.append(normalize_none_history(data,symbol=entry['symbol'],interval='5m',fetched_at=entry['fetched_at']))
raw=pd.concat(parts,ignore_index=True);schedule=xc.get_calendar('XNYS',start=plan['start'],end=plan['as_of']).schedule;coverage=audit_context(raw,schedule,symbols=plan['symbols']);assert raw.is_final.all()
coverage['complete_valid']=coverage.missing_bars.eq(0)&coverage.unexpected_bars.eq(0)&coverage.invalid_rows.eq(0)
provider=root/plan['provider'];fields=['open','high','low','close','volume','turnover','price_valid'];manifest=build_qlib_store_from_frame(raw.rename(columns={'symbol':'instrument','timestamp':'datetime'}),provider_uri=provider,freq='5min',as_of=plan['as_of'],field_map={x:x for x in fields},source='Futu observed NONE RTH constituent5m Q3 snapshot; no imputation or price-plane conversion')
qlib.init(provider_uri=str(provider),region='us');native=load_qlib_feature_frame(QlibProviderConfig(provider,freq='5min'),instruments=plan['symbols'],fields=['$open','$close','$price_valid'],start=plan['start'],end=plan['as_of']+' 16:00:00')
expected=raw.copy();expected['datetime']=expected.timestamp.dt.tz_convert('America/New_York').dt.tz_localize(None);expected=expected.rename(columns={'symbol':'instrument'}).set_index(['instrument','datetime'])
for field in ['open','close','price_valid']:np.testing.assert_allclose(native['$'+field].reindex(expected.index),expected[field],rtol=2e-7,atol=1e-4)
parent=root/plan['daily_card'];ps=json.loads((parent/'summary.json').read_text());mp=parent/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==plan['daily_manifest_sha256'];entry=next(e for e in json.loads(mp.read_text())['entries'] if e['path'].endswith('/artifacts/analysis/prices'))
with R.uri_context(ps['recorder_uri']):
 rec=R.get_recorder(recorder_id=plan['daily_recorder_id'],experiment_name=parent.name);p=Path(rec.download_artifact('analysis/prices'));assert hashlib.sha256(p.read_bytes()).hexdigest()==entry['sha256'];daily=rec.load_object('analysis/prices').reset_index();daily['date']=daily.datetime.dt.strftime('%Y-%m-%d')
marks=raw.sort_values('timestamp').groupby(['symbol','date']).agg(intraday_open=('open','first'),intraday_close=('close','last'));comparison=marks.join(daily.rename(columns={'instrument':'symbol'}).set_index(['symbol','date'])[['$close']]);comparison['close_difference']=comparison.intraday_close-comparison['$close'];assert comparison['$close'].notna().all()
by_symbol=coverage.groupby('symbol').agg(sessions=('date','size'),complete_valid_sessions=('complete_valid','sum'),missing_bars=('missing_bars','sum'),unexpected_bars=('unexpected_bars','sum'),invalid_rows=('invalid_rows','sum'))
by_symbol['maximum_daily_close_difference']=comparison.groupby('symbol').close_difference.apply(lambda x:x.abs().max());by_symbol['daily_close_differences_above_cent']=comparison.groupby('symbol').close_difference.apply(lambda x:x.abs().gt(.0101).sum())
basket=coverage.groupby('date').agg(full_basket=('complete_valid','all'),members=('symbol','size'))
summary={'experiment':card.name,'decision':'constituent_five_minute_source_intake_no_alpha_claim','new_fits':0,'new_backtests':0,'new_orders':0,'objective_achieved':False,'symbols':len(plan['symbols']),'raw_bars':len(raw),'sessions':len(basket),'complete_valid_basket_sessions':int(basket.full_basket.sum()),'missing_bars':int(coverage.missing_bars.sum()),'unexpected_bars':int(coverage.unexpected_bars.sum()),'invalid_rows':int(coverage.invalid_rows.sum()),'source_adjustment':'NONE','session':'RTH','historical_first_receipt_certified':False,'discovery_quarter_already_reused':True,'universe_role':'Union of previously published Q3 baskets; per-decision membership remains date-gated, not all historical constituents','provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/data.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/coverage','analysis/daily_price_comparison','analysis/native']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='read_only_constituent5m_intake',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,workers=1,request_pause_seconds=1.)
 rec.save_objects(artifact_path='inputs',plan=plan,source_manifest=source,raw_sources=raw_sources,rehab={e['symbol']:pd.read_parquet(output/e['path']) for e in source['rehab']},provider_manifest=manifest,quota_before=json.loads((card/'quota_before.json').read_text()))
 rec.save_objects(artifact_path='analysis',coverage=coverage,daily_price_comparison=comparison,native=native,summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')),observed_bars=len(raw))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');by_symbol.to_csv(card/'symbol_coverage.csv');basket.reset_index().to_csv(card/'basket_coverage.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print({k:summary[k] for k in ['raw_bars','complete_valid_basket_sessions','missing_bars','unexpected_bars','invalid_rows']})


# 完整Q3披露篮子五分钟观察：新增来源，不训练

本轮0拟合、0回测、0订单。事先固定2026Q3两份已公开篮子的30个成员并集，拉取Futu NONE/RTH五分钟历史，不按股票表现或日内收益挑选；实际逐日成员权重仍须遵守此前披露日期，不能将并集冒充官方实时指数。

共149760条原始观察、每标的4992条、64个完整交易日；所有成员的原生交易时段网格缺失0、意外时点0、无效OHLC0，64/64日完整篮子可观察。零成交量bar为0条，保留原始数据；完整网格不等于每个bar都有新交易或已认证实时首收。

独立日线来源的1920个标的日收盘核对：最大差0.00501343美元，没有超过1美分的记录；容差包含原生float32误差。该核对只支持收盘尺度一致，不证明每根五分钟OHLC、开盘成交或盘口可执行性。未把日线价格替换进分钟源。

Qlib原生读回开/收盘和有效标记通过。原始SDK数据、取回时间、公司行动响应、逐日覆盖及原生读回均归档NAS，Git保留代码与瘦卡。源码新增可选逐请求暂停，默认0保持旧调用行为；本次单worker每页暂停1秒，断点续传既有回归检查通过。检查点在仓库外。

配额由8已用/92剩余变为36已用/64剩余，新增28个历史符号；只调用报价历史和复权数据接口，没有交易调用。使用技能[Futu数据摄取](../../.codex/skills/futu-api/SKILL.md)的只读边界。

决定：constituent_five_minute_source_intake_no_alpha_claim。完整篮子日内数据是新的观察层，不证明存在可预测同步性或超额。该季度已用于问题发现，不是新的独立评价期；没有计算任何交易规则或挑选预测因子，现役release未改。

下一步先对齐各成员09:45/10:30等已闭合窗口和信息新鲜度，再区分ETF已反映的共同波动与真正不同的内部结构；不将五分钟bar关闭时刻当作已认证实盘接收，不使用尚未闭合bar，不重开已失败的旧日频四特征广度模型。

[逐成员覆盖与日线核对](symbol_coverage.csv) · [全部64日篮子覆盖](basket_coverage.csv) · [固定范围](plan.json)

Recorder `f9a462ad9d4846fe8407bb556b48c844`。
